# Lab 02: Training a Custom Model


**Objective of this lab**: training a small custom model on the Tiny-ImageNet dataset.

## Dataset preparation

In [1]:
"""
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip tiny-imagenet-200.zip -d tiny-imagenet
"""

'\n!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip\n!unzip tiny-imagenet-200.zip -d tiny-imagenet\n'

We need to adjust the format of the val split of the dataset to be used with ImageFolder.

In [2]:
"""
import os
import shutil

with open('tiny-imagenet/tiny-imagenet-200/val/val_annotations.txt') as f:
    for line in f:
        fn, cls, *_ = line.split('\t')
        os.makedirs(f'tiny-imagenet/tiny-imagenet-200/val/{cls}', exist_ok=True)

        shutil.copyfile(f'tiny-imagenet/tiny-imagenet-200/val/images/{fn}', f'tiny-imagenet/tiny-imagenet-200/val/{cls}/{fn}')

shutil.rmtree('tiny-imagenet/tiny-imagenet-200/val/images')
"""

"\nimport os\nimport shutil\n\nwith open('tiny-imagenet/tiny-imagenet-200/val/val_annotations.txt') as f:\n    for line in f:\n        fn, cls, *_ = line.split('\t')\n        os.makedirs(f'tiny-imagenet/tiny-imagenet-200/val/{cls}', exist_ok=True)\n\n        shutil.copyfile(f'tiny-imagenet/tiny-imagenet-200/val/images/{fn}', f'tiny-imagenet/tiny-imagenet-200/val/{cls}/{fn}')\n\nshutil.rmtree('tiny-imagenet/tiny-imagenet-200/val/images')\n"

In [3]:
from torchvision.datasets import ImageFolder
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((64, 64)),  # Resize to fit the input dimensions of the network
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# root/{classX}/x001.jpg

tiny_imagenet_dataset_train = ImageFolder(root='tiny-imagenet/tiny-imagenet-200/train', transform=transform)
tiny_imagenet_dataset_val = ImageFolder(root='tiny-imagenet/tiny-imagenet-200/val', transform=transform)

In [4]:
print(f"Length of train dataset: {len(tiny_imagenet_dataset_train)}")
print(f"Length of val dataset: {len(tiny_imagenet_dataset_val)}")

# The following code also checks the number of samples per class
# from collections import Counter

# class_counts = Counter([target for _, target in tiny_imagenet_dataset_val])
# for class_label, count in class_counts.items():
#     print(f"Class {class_label}: {count} entries")


Length of train dataset: 100000
Length of val dataset: 10000


In [5]:
import torch

train_loader = torch.utils.data.DataLoader(tiny_imagenet_dataset_train, batch_size=128, shuffle=True, num_workers=4)
val_loader = torch.utils.data.DataLoader(tiny_imagenet_dataset_val, batch_size=128, shuffle=False)

In [6]:
# Print number of classes
print(f"Number of classes: {len(tiny_imagenet_dataset_train.classes)}")

Number of classes: 200


## Custom model definition

In [7]:
from torch import nn
import torch.nn.functional as F

# Define the custom neural network
class CustomNet(nn.Module):
    def __init__(self):
        super(CustomNet, self).__init__()

        # --- Block 1 ---
        # Input -> (3, 64, 64)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64) # <-- Batch Norm
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        # Output -> (64, 32, 32)

        # --- Block 2 ---
        # Input -> (64, 32, 32)
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128) # <-- Batch Norm
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        # Output -> (128, 16, 16)

        # --- Block 3 ---
        # Input -> (128, 16, 16)
        self.conv3 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256) # <-- Batch Norm
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)
        # Output -> (256, 8, 8)

        # --- Classifier Head ---
        self.flatten = nn.Flatten()
        
        # Calculate in_features
        in_features = 256 * 8 * 8
        
        self.fc1 = nn.Linear(in_features=in_features, out_features=512)
        self.fc1_relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3) # <-- Dropout (using 0.4)

        self.fc2 = nn.Linear(in_features=512, out_features=200) # 200 classes
        
    def forward(self, x):
        # The sequence is now Conv -> BatchNorm -> ReLU -> Pool
        
        # Block 1
        out = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        # Block 2
        out = self.pool2(self.relu2(self.bn2(self.conv2(out))))
        # Block 3
        out = self.pool3(self.relu3(self.bn3(self.conv3(out))))

        # Classifier
        out = self.flatten(out)
        
        out = self.dropout1(self.fc1_relu(self.fc1(out)))
        out = self.fc2(out) # Output: logits

        return out

In [8]:
from tqdm.notebook import tqdm

def train(epoch, model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(enumerate(train_loader), 
                        desc=f'Train Epoch {epoch}', 
                        total=len(train_loader))

    for batch_idx, (inputs, targets) in progress_bar:
        inputs, targets = inputs.cuda(), targets.cuda()

        # Retrieve outputs
        outputs = model(inputs)

        # Backpropagate
        optimizer.zero_grad()
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        current_loss = running_loss / (batch_idx + 1)
        current_acc = 100. * correct / total
        
        # Update the progress bar description
        progress_bar.set_postfix({'Loss': f'{current_loss:.6f}', 'Acc': f'{current_acc:.2f}%'})

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100. * correct / total
    print(f'Train Epoch: {epoch} Loss: {train_loss:.6f} Acc: {train_accuracy:.2f}%')

In [9]:
# Validation loop
def validate(model, val_loader, criterion):
    model.eval()
    val_loss = 0

    correct, total = 0, 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(val_loader):
            inputs, targets = inputs.cuda(), targets.cuda()

            # Retrieve the outputs
            outputs = model(inputs)

            # Calculate loss
            loss = criterion(outputs, targets)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100. * correct / total

    print(f'Validation Loss: {val_loss:.6f} Acc: {val_accuracy:.2f}%')
    return val_accuracy

## Putting everything together

In [10]:
model = CustomNet().cuda()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=0.01, weight_decay=1e-4)

best_acc = 0

# Run the training process for {num_epochs} epochs
num_epochs = 10
for epoch in range(1, num_epochs + 1):
    train(epoch, model, train_loader, criterion, optimizer)

    # At the end of each training iteration, perform a validation step
    val_accuracy = validate(model, val_loader, criterion)

    # Best validation accuracy
    best_acc = max(best_acc, val_accuracy)

print(f'Best validation accuracy: {best_acc:.2f}%')

Train Epoch 1:   0%|          | 0/782 [00:00<?, ?it/s]

Train Epoch: 1 Loss: 5.911646 Acc: 0.50%
Validation Loss: 5.295092 Acc: 0.54%


Train Epoch 2:   0%|          | 0/782 [00:00<?, ?it/s]

Train Epoch: 2 Loss: 5.173295 Acc: 1.20%
Validation Loss: 5.051462 Acc: 2.05%


Train Epoch 3:   0%|          | 0/782 [00:00<?, ?it/s]

...

Train Epoch: 8 Loss: 4.709681 Acc: 3.72%
Validation Loss: 4.518095 Acc: 5.49%


Train Epoch 9:   0%|          | 0/782 [00:00<?, ?it/s]

Train Epoch: 9 Loss: 4.685349 Acc: 3.73%
Validation Loss: 4.633590 Acc: 4.40%


Train Epoch 10:   0%|          | 0/782 [00:00<?, ?it/s]

Train Epoch: 10 Loss: 4.654156 Acc: 4.07%
Validation Loss: 4.512189 Acc: 5.80%
Best validation accuracy: 5.80%
